# 04 · Synthesis, audio tokens, streaming and evaluation

*Voice & Speech Models*, notebook 4 of 4

This notebook runs speech in both directions and then measures the result. You build a small **synthesiser** that turns text into sound through per-frame pitch targets and a bank of oscillators, much as a vocoder turns an acoustic representation into a waveform. You compress audio into discrete **tokens** with a k-means codebook and measure what the compression loses. You process audio in **streaming chunks** and work out the latency that chunking costs. You score a recogniser with **word and character error rates** computed from your own edit distance. Finally, the pieces join into one auditable experiment, the scaffold for the course capstone.

## What you will build

- `upsample` and `oscillator_bank`: per-frame control tracks turned into a waveform without clicks;
- `assign_codes` and `update_codebook`: the two halves of k-means, which learn an audio-token codebook;
- `stream_frames`: a chunk-by-chunk front end that yields exactly the offline frames and records when each became available;
- `edit_distance` and `error_rate`: the Levenshtein distance and corpus-level word and character error rates.

## Learning objectives

By the end of this notebook you will be able to:

1. trace a text-to-speech pipeline from a text frontend through acoustic targets to a waveform, and explain why phase must be accumulated;
2. quantise audio features with a learned codebook, and relate codebook size, bit rate and reconstruction error;
3. build a streaming front end that matches offline processing exactly, and compute the latency added by chunking and lookahead;
4. compute WER and CER at corpus level, and explain why WER can exceed 100%;
5. assemble an auditable speech experiment: a manifest, a configuration, metrics by condition, latency and a failure analysis.

### How to work through this notebook

Run the cells from top to bottom. When you reach a **✍️ Your turn** cell, type the function in the cell provided and then run the check cell beneath it:

- ✅ means your function matches the reference, and from then on the notebook runs **your** code;
- ✍️ means you have not written it yet, and ❌ means it does not match yet. In both cases the notebook quietly uses a hidden reference version, so you can always keep going and come back later.

Each reference solution sits in a collapsed cell just above the function you type. Cells marked **🧪 Try it** invite you to change one value and run them again.

## Dependencies

- **Tier 1:** NumPy for every calculation and Matplotlib for the plots, plus the standard-library modules `json` and `time`.
- The tiny tensor engine from Base Model Training trains the recogniser in the final section, and the feature pipeline from notebook 01 is reused; both are pasted into hidden recap cells.
- One optional cell, tagged `visualization`, plays the synthesised audio with IPython's audio player when it is available. Nothing later depends on it.
- Nothing is downloaded: all audio is synthesised from text by your own code.

## Setup

Run this cell first. It imports NumPy and Matplotlib, creates a seeded random generator and defines `practice`, the helper that checks each function you type against a reference.

In [ ]:
# ── Setup ─ run this cell first ────────────────────────────────────────────
import numpy as np

rng = np.random.default_rng(0)


def _close(got, want, atol):
    if isinstance(want, dict):
        return isinstance(got, dict) and got.keys() == want.keys() and all(
            _close(got[key], want[key], atol) for key in want
        )
    if isinstance(want, (list, tuple)):
        return isinstance(got, (list, tuple)) and len(got) == len(want) and all(
            _close(g, w, atol) for g, w in zip(got, want)
        )
    try:
        got_array, want_array = np.asarray(got, dtype=float), np.asarray(want, dtype=float)
    except (TypeError, ValueError):
        return got == want
    return got_array.shape == want_array.shape and np.allclose(got_array, want_array, atol=atol)


def _short(value):
    if isinstance(value, tuple):
        return "(" + ", ".join(_short(item) for item in value) + ")"
    if isinstance(value, np.ndarray):
        text = np.array2string(value, precision=4, threshold=8, separator=", ").replace("\n", "")
    else:
        text = repr(value)
    return text if len(text) < 120 else text[:117] + "..."


def practice(name, yours, reference, cases, atol=1e-6):
    """Check your function against the reference on a few inputs.

    Returns the function the rest of the notebook should use: yours once it
    matches, otherwise the reference -- so you can always keep going.
    """
    try:
        for args in cases:
            got, want = yours(*args), reference(*args)
            if not _close(got, want, atol):
                print(f"❌ {name}: for input {_short(args)}")
                print(f"   you returned {_short(got)}")
                print(f"   expected     {_short(want)}")
                print(f"   The rest of the notebook uses the reference {name} until yours matches.")
                return reference
    except NotImplementedError:
        print(f"✍️  {name}: not written yet. Type it in the cell above and run this cell again.")
        print("   Until then the rest of the notebook uses the reference version.")
        return reference
    except Exception as error:  # a typo in the learner's code must not stop the lesson
        print(f"❌ {name} raised {type(error).__name__}: {error}")
        print(f"   The rest of the notebook uses the reference {name} until yours runs.")
        return reference
    print(f"✅ {name}: matches the reference on {len(cases)} test case{'s' if len(cases) != 1 else ''}. From here on the notebook runs YOUR {name}.")
    return yours

import json
import time

import matplotlib.pyplot as plt

## 📦 Recap: the tiny tensor engine you built in Base Model Training

The hidden cell below is the reverse-mode autograd engine from Base Model Training, unchanged: `Tensor` (with `@`, broadcasting arithmetic, `relu`, `tanh`, `log_softmax`, `softmax`, indexing and `backward`), `parameter`, `zeros_parameter`, `zero_grad`, `cross_entropy`, `layer_norm`, `numerical_grad` and `relative_error`. Every operation in it has been gradient-checked. Run the cell; open it only if you want to look at the code again.

In [ ]:
# ── Tiny tensor autograd (NumPy) ── the engine you built in Base Model Training ──
import numpy as np


def _unbroadcast(grad, shape):
    """Sum a gradient back down to `shape`, undoing NumPy broadcasting."""
    while grad.ndim > len(shape):
        grad = grad.sum(axis=0)
    for axis, size in enumerate(shape):
        if size == 1 and grad.shape[axis] != 1:
            grad = grad.sum(axis=axis, keepdims=True)
    return grad


class Tensor:
    """A NumPy array that remembers how it was made, so gradients can flow back."""

    def __init__(self, data, requires_grad=False, parents=(), backward=None, op=""):
        self.data = np.asarray(data, dtype=np.float64)
        self.requires_grad = requires_grad
        self.grad = None
        self._parents = parents
        self._backward = backward  # grad_out -> one gradient per parent
        self.op = op

    shape = property(lambda self: self.data.shape)
    ndim = property(lambda self: self.data.ndim)
    T = property(lambda self: self.transpose())

    def __repr__(self):
        return f"Tensor(shape={self.shape}, op={self.op or 'leaf'}, requires_grad={self.requires_grad})"

    @staticmethod
    def lift(value):
        return value if isinstance(value, Tensor) else Tensor(value)

    def _result(self, data, parents, backward, op):
        if any(parent.requires_grad for parent in parents):
            return Tensor(data, True, parents, backward, op)
        return Tensor(data, op=op)

    # arithmetic, with broadcasting
    def __add__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data + other.data, (self, other), lambda g: (g, g), "add")

    __radd__ = __add__

    def __neg__(self):
        return self._result(-self.data, (self,), lambda g: (-g,), "neg")

    def __sub__(self, other):
        return self + (-Tensor.lift(other))

    def __rsub__(self, other):
        return Tensor.lift(other) + (-self)

    def __mul__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data * other.data, (self, other),
                            lambda g: (g * other.data, g * self.data), "mul")

    __rmul__ = __mul__

    def __truediv__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data / other.data, (self, other),
                            lambda g: (g / other.data, -g * self.data / other.data ** 2), "div")

    def __rtruediv__(self, other):
        return Tensor.lift(other) / self

    def __pow__(self, exponent):
        assert isinstance(exponent, (int, float)), "only constant exponents"
        return self._result(self.data ** exponent, (self,),
                            lambda g: (g * exponent * self.data ** (exponent - 1),), "pow")

    def __matmul__(self, other):
        other = Tensor.lift(other)

        def backward(g):
            return (g @ np.swapaxes(other.data, -1, -2), np.swapaxes(self.data, -1, -2) @ g)

        return self._result(self.data @ other.data, (self, other), backward, "matmul")

    # elementwise functions
    def exp(self):
        out = np.exp(self.data)
        return self._result(out, (self,), lambda g: (g * out,), "exp")

    def log(self):
        return self._result(np.log(self.data), (self,), lambda g: (g / self.data,), "log")

    def tanh(self):
        out = np.tanh(self.data)
        return self._result(out, (self,), lambda g: (g * (1 - out ** 2),), "tanh")

    def relu(self):
        return self._result(np.maximum(self.data, 0), (self,), lambda g: (g * (self.data > 0),), "relu")

    def sigmoid(self):
        out = 0.5 * (1 + np.tanh(0.5 * self.data))  # stable for large |x|
        return self._result(out, (self,), lambda g: (g * out * (1 - out),), "sigmoid")

    # reductions and reshaping
    def sum(self, axis=None, keepdims=False):
        def backward(g):
            if axis is not None and not keepdims:
                g = np.expand_dims(g, axis)
            return (np.broadcast_to(g, self.shape).copy(),)

        return self._result(self.data.sum(axis=axis, keepdims=keepdims), (self,), backward, "sum")

    def mean(self, axis=None, keepdims=False):
        count = self.data.size if axis is None else np.prod([self.shape[a] for a in np.atleast_1d(axis)])
        return self.sum(axis=axis, keepdims=keepdims) / count

    def reshape(self, *shape):
        return self._result(self.data.reshape(*shape), (self,), lambda g: (g.reshape(self.shape),), "reshape")

    def transpose(self, *axes):
        axes = axes or tuple(reversed(range(self.ndim)))
        inverse = np.argsort(axes)
        return self._result(self.data.transpose(axes), (self,), lambda g: (g.transpose(inverse),), "transpose")

    def __getitem__(self, index):
        def backward(g):
            full = np.zeros_like(self.data)
            np.add.at(full, index, g)  # repeated indices (an embedding row used twice) add up
            return (full,)

        return self._result(self.data[index], (self,), backward, "index")

    def log_softmax(self, axis=-1):
        shifted = self.data - self.data.max(axis=axis, keepdims=True)
        out = shifted - np.log(np.exp(shifted).sum(axis=axis, keepdims=True))
        probs = np.exp(out)
        return self._result(out, (self,), lambda g: (g - probs * g.sum(axis=axis, keepdims=True),), "log_softmax")

    def softmax(self, axis=-1):
        shifted = np.exp(self.data - self.data.max(axis=axis, keepdims=True))
        out = shifted / shifted.sum(axis=axis, keepdims=True)
        return self._result(out, (self,), lambda g: (out * (g - (g * out).sum(axis=axis, keepdims=True)),), "softmax")

    # reverse-mode differentiation
    def backward(self, grad=None):
        order, seen = [], set()

        def visit(node):
            if id(node) not in seen:
                seen.add(id(node))
                for parent in node._parents:
                    visit(parent)
                order.append(node)

        visit(self)
        self.grad = np.ones_like(self.data) if grad is None else np.asarray(grad, dtype=np.float64)
        for node in reversed(order):
            if node._backward is None or node.grad is None:
                continue
            for parent, parent_grad in zip(node._parents, node._backward(node.grad)):
                if not parent.requires_grad or parent_grad is None:
                    continue
                parent_grad = _unbroadcast(np.asarray(parent_grad, dtype=np.float64), parent.shape)
                parent.grad = parent_grad if parent.grad is None else parent.grad + parent_grad


def parameter(*shape, rng, scale=None):
    """A trainable weight, initialised with N(0, scale^2); default scale is He's sqrt(2 / fan_in)."""
    scale = np.sqrt(2.0 / shape[0]) if scale is None else scale
    return Tensor(rng.normal(0.0, scale, shape), requires_grad=True)


def zeros_parameter(*shape):
    return Tensor(np.zeros(shape), requires_grad=True)


def zero_grad(params):
    for p in params:
        p.grad = None


def cross_entropy(logits, targets, mask=None):
    """Mean next-token/class cross-entropy. logits: (..., classes); targets: int (...); mask: 1.0 keeps a position."""
    flat = logits.reshape(-1, logits.shape[-1]).log_softmax(axis=-1)
    targets = np.asarray(targets).reshape(-1)
    picked = flat[np.arange(targets.size), targets]
    if mask is None:
        return -picked.mean()
    weights = np.asarray(mask, dtype=np.float64).reshape(-1)
    return -(picked * weights).sum() / max(weights.sum(), 1.0)


def layer_norm(x, gamma, beta, eps=1e-5):
    mean = x.mean(axis=-1, keepdims=True)
    centred = x - mean
    var = (centred ** 2).mean(axis=-1, keepdims=True)
    return centred / (var + eps) ** 0.5 * gamma + beta


def numerical_grad(f, tensor, eps=1e-6):
    """Central-difference gradient of scalar f() with respect to tensor.data."""
    grad = np.zeros_like(tensor.data)
    it = np.nditer(tensor.data, flags=["multi_index"])
    for _ in it:
        index = it.multi_index
        original = tensor.data[index]
        tensor.data[index] = original + eps
        plus = float(f().data)
        tensor.data[index] = original - eps
        minus = float(f().data)
        tensor.data[index] = original
        grad[index] = (plus - minus) / (2 * eps)
    return grad


def relative_error(a, b):
    return float(np.linalg.norm(a - b) / max(np.linalg.norm(a) + np.linalg.norm(b), 1e-12))

## 📦 Recap: what you built earlier

The hidden cell below defines the feature pipeline from notebook 01 under its usual names: `make_tone`, `hann_window`, `frame_signal`, `stft_power`, `hz_to_mel`, `mel_to_hz`, `mel_filterbank`, `log_mel` and `log_mel_spectrogram`. Run it; open it only if you want to look back at the code.

In [ ]:
# 📦 Recap: the pieces from notebook 01, under their usual names.

def make_tone(frequency, duration, sample_rate, amplitude=1.0, phase=0.0):
    """A sampled sine wave of shape (int(round(duration * sample_rate)),)."""
    n_samples = int(round(duration * sample_rate))
    t = np.arange(n_samples) / sample_rate
    return amplitude * np.sin(2 * np.pi * frequency * t + phase)


def hann_window(length):
    """The periodic Hann window: 0.5 - 0.5 cos(2 pi n / length) for n = 0, ..., length - 1."""
    return 0.5 - 0.5 * np.cos(2 * np.pi * np.arange(length) / length)


def frame_signal(x, frame_length, hop):
    """Overlapping frames of x: shape (1 + (len(x) - frame_length) // hop, frame_length)."""
    n_frames = 1 + (len(x) - frame_length) // hop
    return np.array([x[m * hop: m * hop + frame_length] for m in range(n_frames)])


def stft_power(x, frame_length, hop):
    """Power spectrogram |rfft(window * frame)|^2: shape (M, frame_length // 2 + 1)."""
    frames = frame_signal(x, frame_length, hop) * hann_window(frame_length)
    return np.abs(np.fft.rfft(frames, axis=1)) ** 2


def hz_to_mel(frequency):
    """HTK mel scale."""
    return 2595.0 * np.log10(1.0 + np.asarray(frequency, dtype=float) / 700.0)


def mel_to_hz(mel):
    """Inverse of hz_to_mel."""
    return 700.0 * (10.0 ** (np.asarray(mel, dtype=float) / 2595.0) - 1.0)


def mel_filterbank(n_mels, n_fft, sample_rate, f_min=0.0, f_max=None):
    """Triangular mel filters: shape (n_mels, n_fft // 2 + 1)."""
    f_max = sample_rate / 2 if f_max is None else f_max
    edges = mel_to_hz(np.linspace(hz_to_mel(f_min), hz_to_mel(f_max), n_mels + 2))
    freqs = np.arange(n_fft // 2 + 1) * sample_rate / n_fft
    left, centre, right = edges[:-2, None], edges[1:-1, None], edges[2:, None]
    rising = (freqs - left) / (centre - left)
    falling = (right - freqs) / (right - centre)
    return np.maximum(0.0, np.minimum(rising, falling))


def log_mel(power, filterbank, eps=1e-10):
    """Log band energies: (M, n_bins) power and (n_mels, n_bins) filters give (M, n_mels)."""
    return np.log(power @ filterbank.T + eps)


def log_mel_spectrogram(x, sample_rate, frame_length=256, hop=128, n_mels=20):
    """Waveform (N,) -> log-mel features (M, n_mels)."""
    return log_mel(stft_power(x, frame_length, hop), mel_filterbank(n_mels, frame_length, sample_rate))

## 1 · From text to sound

A text-to-speech system has three stages:

1. a **text frontend** turns the written text into units to pronounce (normalising numbers and abbreviations, often converting letters to phonemes);
2. an **acoustic model** predicts, frame by frame, how those units should sound: durations, pitch, loudness, or a whole spectrogram;
3. a **vocoder** turns the acoustic representation into waveform samples.

Here each stage is as small as it can be. The "language" has three letters, **a**, **b** and **c**, and the frontend maps every letter to a tone. The acoustic model is a lookup table: each letter lasts about six frames (96 ms) at its own pitch, letters within a word are separated by a short pause, and words by a longer one. The output is two **control tracks**, a pitch $f_0$ in hertz and an amplitude for every 16 ms frame, together with the letter that each frame belongs to. Because we generate the speech ourselves, we know that alignment exactly, which will let us train a recogniser in section 5 without any manual labelling. A voice is described by a pitch factor and a speaking rate.

In [ ]:
SAMPLE_RATE, FRAME_LENGTH, HOP, N_MELS = 8000, 256, 128, 20
LETTER_HZ = {"a": 500.0, "b": 850.0, "c": 1400.0}
LETTERS = " abc"                                   # index 0 means "no letter": silence
LEXICON = ["ab", "ba", "cab", "abc", "cc", "bca", "acb", "aa"]


def text_to_controls(text, rng, pitch=1.0, rate=1.0):
    """Frontend + acoustic model: per-frame pitch (Hz), amplitude and letter index for a text."""
    f0, amplitude, labels = [], [], []

    def hold(n_frames, hz, amp, label):
        f0.extend([hz] * n_frames); amplitude.extend([amp] * n_frames); labels.extend([label] * n_frames)

    hold(int(round(4 * rate)), 0.0, 0.0, 0)                              # lead-in silence
    for w, word in enumerate(text.split()):
        if w > 0:
            hold(int(round(10 * rate)), 0.0, 0.0, 0)                     # pause between words
        for i, letter in enumerate(word):
            if i > 0:
                hold(int(round(3 * rate)), 0.0, 0.0, 0)                  # short pause between letters
            hold(int(round(6 * rate * rng.uniform(0.8, 1.2))), LETTER_HZ[letter] * pitch,
                 rng.uniform(0.4, 0.8), LETTERS.index(letter))
    hold(int(round(4 * rate)), 0.0, 0.0, 0)                              # trailing silence
    f0, amplitude, labels = np.array(f0), np.array(amplitude), np.array(labels)
    voiced = np.flatnonzero(f0 > 0)
    f0 = f0[voiced][np.maximum(np.searchsorted(voiced, np.arange(len(f0)), side="right") - 1, 0)]
    return f0, amplitude, labels


demo_f0, demo_amp, demo_labels = text_to_controls("cab ba", np.random.default_rng(0))
print(f"'cab ba' -> {len(demo_f0)} frames = {len(demo_f0) * HOP / SAMPLE_RATE:.2f} s")
print("letters per frame:", "".join(LETTERS[k] if k else "." for k in demo_labels))

**What to notice.** During pauses the pitch track keeps the previous letter's pitch (the first pause takes the first letter's) while the amplitude is zero. The oscillator then never jumps in frequency during silence, so it cannot click there.

### Upsampling the controls

The control tracks have one value per 16 ms frame, but the waveform needs one value per sample: 128 per frame at 8 kHz. Holding each value for 128 samples would make the pitch and loudness jump at every frame boundary; interpolating linearly between frames makes them glide. If frame $j$'s value sits at sample position $j \cdot H$, sample $n$ sits at frame position $n / H$, so

$$v[n] = \operatorname{interp}\!\left(\frac{n}{H};\; j = 0, 1, \dots, J - 1;\; v_0, \dots, v_{J-1}\right), \qquad n = 0, \dots, J H - 1,$$

where `np.interp` holds the last value beyond the final frame.

### ✍️ Your turn: `upsample`

Write `upsample(values, hop)` that returns an array of `len(values) * hop` samples, interpolating linearly as above: `np.interp(np.arange(len(values) * hop) / hop, np.arange(len(values)), values)`.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def upsample_reference(values, hop):
    return np.interp(np.arange(len(values) * hop) / hop, np.arange(len(values)), values)

In [ ]:
def upsample(values, hop):
    """Per-frame values (J,) -> per-sample values (J * hop,), linearly interpolated."""
    # ✍️ Your code here (one line)
    raise NotImplementedError

In [ ]:
upsample = practice("upsample", upsample, upsample_reference, cases=[
    (np.array([0.0, 1.0]), 4),           # [0, 0.25, 0.5, 0.75, 1, 1, 1, 1]
    (np.array([5.0]), 3),                # a single frame is held
    (np.array([100.0, 200.0, 50.0]), 2),
])
print(upsample(np.array([0.0, 1.0]), 4))

### An oscillator bank

To play a pitch that changes over time, an oscillator must **accumulate** its phase: the phase advances by $2\pi f_0[n] / f_s$ radians at every sample,

$$\phi[n] = \frac{2\pi}{f_s} \sum_{m=0}^{n} f_0[m], \qquad y[n] = A[n] \sum_{k=1}^{K} h_k \sin\!\big(k\, \phi[n]\big),$$

with harmonic weights $h_1, \dots, h_K$ giving the voice its colour. Writing $\sin(2\pi f_0[n]\, n / f_s)$ instead looks similar but is wrong: when $f_0$ changes, the phase jumps, and every jump is an audible click. The demo shows the jump: three tones joined end to end, first restarted from zero phase at each change, then with an accumulated phase.

In [ ]:
segments = [(500.0, 0.093), (850.0, 0.107), (1400.0, 0.081)]
restarted = np.concatenate([make_tone(f, d, SAMPLE_RATE) for f, d in segments])
f0_track = np.concatenate([np.full(int(round(d * SAMPLE_RATE)), f) for f, d in segments])
accumulated = np.sin(2 * np.pi * np.cumsum(f0_track) / SAMPLE_RATE)
boundary = int(round(0.093 * SAMPLE_RATE))


def splatter(x):
    """Fraction of the power above 2 kHz, where none of these tones has any energy."""
    power_spectrum = stft_power(x, FRAME_LENGTH, HOP)
    return power_spectrum[:, np.arange(FRAME_LENGTH // 2 + 1) * SAMPLE_RATE / FRAME_LENGTH > 2000].sum() / power_spectrum.sum()


print(f"power above 2 kHz: restarted phase {splatter(restarted):.1e}, accumulated phase {splatter(accumulated):.1e}")
window = slice(boundary - 12, boundary + 12)
plt.figure(figsize=(8, 2.6))
plt.plot(np.arange(len(restarted))[window], restarted[window], "o-", ms=3, label="phase restarted at the change")
plt.plot(np.arange(len(accumulated))[window], accumulated[window], "o-", ms=3, label="phase accumulated")
plt.axvline(boundary, color="0.6", lw=0.8)
plt.xlabel("sample"); plt.title("500 Hz becomes 850 Hz: a jump in the waveform, or a smooth change"); plt.legend(fontsize=8)
plt.tight_layout(); plt.show()

### ✍️ Your turn: `oscillator_bank`

Write `oscillator_bank(f0, amplitude, sample_rate, harmonics=(1.0, 0.5, 0.25))`:

- **inputs:** per-sample pitch `f0` and `amplitude`, both of shape `(N,)`, and the harmonic weights $h_1, h_2, \dots$;
- **output:** the waveform $y$ of shape `(N,)` from the formulas above.

Hints: `phase = 2 * np.pi * np.cumsum(f0) / sample_rate`; then add up `weight * np.sin((k + 1) * phase)` over `enumerate(harmonics)` and multiply by the amplitude.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def oscillator_bank_reference(f0, amplitude, sample_rate, harmonics=(1.0, 0.5, 0.25)):
    phase = 2 * np.pi * np.cumsum(f0) / sample_rate
    return amplitude * sum(weight * np.sin((k + 1) * phase) for k, weight in enumerate(harmonics))

In [ ]:
def oscillator_bank(f0, amplitude, sample_rate, harmonics=(1.0, 0.5, 0.25)):
    """A harmonic oscillator with an accumulated phase: per-sample f0 and amplitude -> waveform (N,)."""
    # ✍️ Your code here (two lines)
    raise NotImplementedError

In [ ]:
osc_rng = np.random.default_rng(1)
oscillator_bank = practice("oscillator_bank", oscillator_bank, oscillator_bank_reference, cases=[
    (np.full(16, 1000.0), np.ones(16), 8000),
    (np.linspace(300.0, 900.0, 40), osc_rng.uniform(size=40), 8000, (1.0,)),
    (np.full(5, 440.0), np.zeros(5), 16000, (1.0, 0.3)),     # zero amplitude: silence
])

The vocoder is now two lines: upsample the controls and run the oscillator bank. The function `speak` below is the whole synthesiser, from text to waveform, with optional noise at a chosen signal-to-noise ratio. The check estimates the pitch of every voiced frame from the synthesised audio and compares it with the target.

In [ ]:
def synthesise(f0_frames, amp_frames, hop=HOP, sample_rate=SAMPLE_RATE):
    """Vocoder: per-frame controls -> waveform, with your upsample and oscillator_bank."""
    return oscillator_bank(upsample(f0_frames, hop), upsample(amp_frames, hop), sample_rate)


def speak(text, rng, pitch=1.0, rate=1.0, snr_db=None):
    """Text -> (waveform, letter index of every analysis frame). Noise is mixed in at snr_db if given."""
    f0, amplitude, labels = text_to_controls(text, rng, pitch, rate)
    x = synthesise(f0, amplitude)
    if snr_db is not None:
        noise = rng.normal(size=len(x))
        x = x + noise * np.sqrt(np.mean(x ** 2) / (np.mean(noise ** 2) * 10 ** (snr_db / 10)))
    n_frames = 1 + (len(x) - FRAME_LENGTH) // HOP
    centres = (np.arange(n_frames) * HOP + FRAME_LENGTH // 2) // HOP     # the control frame at each window's centre
    return x, labels[np.minimum(centres, len(labels) - 1)]


demo_wave, demo_frame_labels = speak("cab ba", np.random.default_rng(0))
power_frames = stft_power(demo_wave, FRAME_LENGTH, HOP)
estimated = np.argmax(power_frames, axis=1) * SAMPLE_RATE / FRAME_LENGTH
targets = np.array([LETTER_HZ[LETTERS[k]] if k else 0.0 for k in demo_frame_labels])
voiced = demo_frame_labels > 0
pitch_error = np.abs(estimated[voiced] - targets[voiced])
print(f"{voiced.sum()} voiced frames; largest pitch error {pitch_error.max():.1f} Hz (one FFT bin is {SAMPLE_RATE / FRAME_LENGTH:.2f} Hz)")

fig, axes = plt.subplots(2, 1, figsize=(8, 4.2), sharex=True)
axes[0].imshow(log_mel_spectrogram(demo_wave, SAMPLE_RATE).T, origin="lower", aspect="auto", cmap="magma")
axes[0].set_ylabel("mel band"); axes[0].set_title("Synthesised 'cab ba': log-mel spectrogram and the letters behind it")
axes[1].plot(demo_frame_labels, drawstyle="steps-mid"); axes[1].set_yticks(range(4)); axes[1].set_yticklabels(["pause", "a", "b", "c"])
axes[1].set_xlabel("frame")
plt.tight_layout(); plt.show()

In [ ]:
try:
    from IPython.display import Audio, display
except ImportError:
    print("IPython's audio player is not available here; the spectrogram shows the same sound.")
else:
    display(Audio(demo_wave, rate=SAMPLE_RATE))

**Honest scope.** Real TTS systems learn their acoustic model from recorded speech (predicting mel spectrograms or durations, pitch and energy), and their vocoders are neural networks trained to produce natural waveforms from those representations. The structure is the same as here: a frontend whose policy decides how "12/03" is read, an acoustic model that must get timing and prosody right, and a vocoder that must not click. A low reconstruction loss is no guarantee of natural speech, which is why synthesis is evaluated by listening.

## 2 · Audio tokens with a k-means codebook

Language models work on discrete tokens. To model audio the same way, we can **quantise** it: learn a **codebook** of $K$ vectors and replace every feature frame by the index of its nearest code vector. A sequence of frames becomes a sequence of integers between 0 and $K - 1$, and a decoder can turn the integers back into code vectors. Neural audio codecs learn the codebooks and the decoder jointly and often stack several codebooks, but the idea is the same.

The classic way to learn a codebook is **k-means** (Lloyd's algorithm). Start from $K$ frames chosen at random, then repeat two steps:

1. **assign** every frame $x_m$ to its nearest code, $\text{code}(m) = \arg\min_j \lVert x_m - c_j \rVert^2$;
2. **update** every code vector to the mean of the frames assigned to it; a code with no frames keeps its previous vector.

Each step can only lower the total squared distance, so the error never increases. Shapes: frames `(M, D)`, codebook `(K, D)`, codes `(M,)`. Below, the frames are the 20-dimensional log-mel frames of 150 synthesised utterances.

In [ ]:
token_rng = np.random.default_rng(1)
token_texts = [" ".join(token_rng.choice(LEXICON, size=int(token_rng.integers(1, 4)))) for _ in range(150)]
token_frames = np.concatenate([log_mel_spectrogram(speak(t, token_rng, snr_db=20)[0], SAMPLE_RATE) for t in token_texts])
print("frames to quantise:", token_frames.shape)
distances_demo = ((token_frames[:3, None, :] - token_frames[None, :2, :]) ** 2).sum(axis=-1)
print("squared distances from 3 frames to 2 candidate codes, shape", distances_demo.shape)

### ✍️ Your turn: `assign_codes`

Write `assign_codes(frames, codebook)` that returns, for every frame, the index of the nearest code vector by squared Euclidean distance, as an integer array of shape `(M,)`. If two codes are equally near, take the lower index, which is what `np.argmin` does.

Hint: broadcasting `frames[:, None, :] - codebook[None, :, :]` gives all the differences at once, shape `(M, K, D)`.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def assign_codes_reference(frames, codebook):
    distances = ((frames[:, None, :] - codebook[None, :, :]) ** 2).sum(axis=-1)
    return np.argmin(distances, axis=1)

In [ ]:
def assign_codes(frames, codebook):
    """Index of the nearest code vector for every frame: (M, D) and (K, D) -> (M,) integers."""
    # ✍️ Your code here (two lines)
    raise NotImplementedError

In [ ]:
assign_codes = practice("assign_codes", assign_codes, assign_codes_reference, cases=[
    (np.array([[0.0, 0.0], [1.0, 1.0], [0.9, 0.2]]), np.array([[0.0, 0.0], [1.0, 0.0], [1.0, 1.0]])),
    (np.array([[0.5, 0.0]]), np.array([[0.0, 0.0], [1.0, 0.0]])),      # a tie: the lower index wins
    (token_frames[:50], token_frames[100:108]),
])

### ✍️ Your turn: `update_codebook`

Write `update_codebook(frames, codes, codebook)` that returns a **new** codebook of shape `(K, D)`: row `j` is the mean of the frames whose code is `j`, or the old row `j` if no frame has that code. Do not modify the codebook you were given.

Hint: start from `codebook.copy()` and loop over `j in range(len(codebook))`.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def update_codebook_reference(frames, codes, codebook):
    updated = codebook.copy()
    for j in range(len(codebook)):
        members = frames[codes == j]
        if len(members) > 0:
            updated[j] = members.mean(axis=0)
    return updated

In [ ]:
def update_codebook(frames, codes, codebook):
    """Move every code vector to the mean of its frames; codes without frames stay where they are."""
    # ✍️ Your code here (about five lines)
    raise NotImplementedError

In [ ]:
update_codebook = practice("update_codebook", update_codebook, update_codebook_reference, cases=[
    (np.array([[0.0, 0.0], [2.0, 0.0], [5.0, 5.0]]), np.array([0, 0, 1]), np.zeros((2, 2))),
    (np.array([[1.0, 1.0]]), np.array([0]), np.array([[0.0, 0.0], [9.0, 9.0]])),   # code 1 is empty: it stays
])

With both halves written, k-means is a short loop. The **relative reconstruction error** compares the squared error of replacing every frame by its code vector with the variance of the frames: 0 means nothing is lost, 1 means the codes are no better than the overall mean. The **bit rate** of the token stream is frames per second times $\log_2 K$ bits per token.

In [ ]:
def kmeans(frames, k, iterations, rng):
    """Lloyd's algorithm from k distinct random frames: returns (codebook, codes)."""
    codebook = frames[rng.choice(len(frames), size=k, replace=False)].copy()
    for _ in range(iterations):
        codebook = update_codebook(frames, assign_codes(frames, codebook), codebook)
    return codebook, assign_codes(frames, codebook)


def relative_reconstruction_error(frames, codebook, codes):
    return float(((frames - codebook[codes]) ** 2).sum(axis=1).mean() / ((frames - frames.mean(axis=0)) ** 2).sum(axis=1).mean())


frames_per_second = SAMPLE_RATE / HOP
codebook_sizes = [2, 4, 8, 16, 32]   # 🧪 try adding 64 or 128, then run this cell again
token_errors = {}
for k in codebook_sizes:
    codebook, codes = kmeans(token_frames, k, iterations=15, rng=np.random.default_rng(0))
    token_errors[k] = relative_reconstruction_error(token_frames, codebook, codes)
    print(f"K = {k:3d}: relative error {token_errors[k]:.3f}, {frames_per_second * np.log2(k):5.0f} bit/s")
print(f"for comparison, 16-bit PCM at 8 kHz is {SAMPLE_RATE * 16:,} bit/s")

codebook16, codes16 = kmeans(token_frames, 16, iterations=15, rng=np.random.default_rng(0))
first = log_mel_spectrogram(speak("cab ba", np.random.default_rng(0), snr_db=20)[0], SAMPLE_RATE)
print("tokens for 'cab ba':", " ".join(str(c) for c in assign_codes(first, codebook16)))

**What to notice.** The error falls quickly as the codebook grows and then flattens: a few codes capture the three tones and silence, and later codes split hairs between noise patterns. Even 32 codes cost only about 310 bits per second against 128,000 for raw 16-bit audio, because a code index cannot describe the fine detail of the waveform; here we only quantise log-mel features, so the phase is gone entirely. The token sequence repeats the same few indices through each letter and pause, which is exactly the structure a sequence model over tokens learns. Real codecs run at 50 to 75 frames per second with several codebooks of about 1,024 entries each.

## 3 · Streaming: frames from chunks, and what they cost in time

A live system does not get the whole recording at once. Audio arrives in **chunks** of, say, 80 ms, and the front end must produce frames as soon as it can. A frame of $L$ samples is complete only when its last sample has arrived, so the front end keeps a **buffer** of samples that have arrived but not yet been used by every frame that needs them:

1. append the new chunk to the buffer;
2. while the buffer holds at least $L$ samples, emit its first $L$ samples as a frame and drop the first $H$ samples (the next frame starts $H$ samples later).

Done correctly, the streamed frames are **identical** to the offline frames, so a model trained offline sees exactly the same input. The time a frame waits for its chunk is part of the system's latency.

### ✍️ Your turn: `stream_frames`

Write `stream_frames(chunks, frame_length, hop)`:

- **input:** a list of 1-D arrays (the chunks, in arrival order; they may have different lengths, and some may be empty);
- **output:** a tuple `(frames, ready)`: `frames` of shape `(M, frame_length)`, exactly as `frame_signal` would produce from the concatenated audio, and `ready`, an integer array of shape `(M,)` in which `ready[i]` is the index of the chunk whose arrival completed frame `i`.

Hints: start with `buffer = np.zeros(0)` and empty lists; `np.concatenate([buffer, chunk])` appends; build the result with `np.array(frames).reshape(-1, frame_length)` so that it has the right shape even when no frame was completed.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def stream_frames_reference(chunks, frame_length, hop):
    buffer, frames, ready = np.zeros(0), [], []
    for index, chunk in enumerate(chunks):
        buffer = np.concatenate([buffer, chunk])
        while len(buffer) >= frame_length:
            frames.append(buffer[:frame_length].copy())
            ready.append(index)
            buffer = buffer[hop:]
    return np.array(frames).reshape(-1, frame_length), np.array(ready, dtype=int)

In [ ]:
def stream_frames(chunks, frame_length, hop):
    """Frames from audio arriving in chunks: (frames (M, frame_length), ready (M,) chunk indices)."""
    buffer, frames, ready = np.zeros(0), [], []
    # ✍️ Your code here (a loop over the chunks with a loop inside it)
    raise NotImplementedError

In [ ]:
ramp = np.arange(12.0)
stream_frames = practice("stream_frames", stream_frames, stream_frames_reference, cases=[
    ([ramp[:5], ramp[5:9], ramp[9:]], 4, 2),
    ([ramp[:3], np.zeros(0), ramp[3:]], 4, 4),          # an empty chunk changes nothing
    ([ramp[:2]], 4, 2),                                 # not enough audio for a single frame
    ([ramp], 4, 3),                                     # everything in one chunk
])
print(stream_frames([ramp[:5], ramp[5:9], ramp[9:]], 4, 2))

Check the promise on a whole synthesised utterance, for several chunk sizes, and then do the latency arithmetic. Frame $i$ ends at sample $iH + L$; it is ready when chunk `ready[i]` has fully arrived, at sample $(\text{ready}[i] + 1)\,C$ for chunks of $C$ samples arriving in real time. The difference is the **chunk wait**, between 0 and $C$ samples. The recogniser in section 5 also looks one frame ahead, so a decision about frame $i$ waits for frame $i + 1$: one more hop of **lookahead**.

In [ ]:
stream_wave, _ = speak("abc cab", np.random.default_rng(3))
offline = frame_signal(stream_wave, FRAME_LENGTH, HOP)
for chunk_ms in (10, 80, 200):
    chunk = int(chunk_ms * SAMPLE_RATE / 1000)
    chunks = [stream_wave[i:i + chunk] for i in range(0, len(stream_wave), chunk)]
    streamed, ready = stream_frames(chunks, FRAME_LENGTH, HOP)
    assert np.array_equal(streamed, offline), "streamed frames must equal the offline frames"
    frame_end = np.arange(len(streamed)) * HOP + FRAME_LENGTH
    wait_ms = ((ready + 1) * chunk - frame_end) / SAMPLE_RATE * 1000
    assert wait_ms.min() >= 0 and wait_ms.max() < chunk_ms, "a frame waits at most one chunk"
    print(f"{chunk_ms:3d} ms chunks: {len(chunks):3d} chunks, frames identical to offline; "
          f"chunk wait {wait_ms.mean():5.1f} ms on average, {wait_ms.max():5.1f} ms at most")

**A latency budget.** With 80 ms chunks, 32 ms frames, a 16 ms hop and one frame of lookahead, the audio at the *start* of a frame is already 32 ms old when the frame is complete; the frame then waits up to 64 ms for its chunk, and the decision waits 16 ms more for the next frame. Add the time to compute the model on each chunk, and for a final transcript the **endpointer**'s wait: the silence it requires (often 300 ms or more) before it declares that the speaker has finished. Halving the chunk size halves the chunk wait but doubles the number of model calls; reducing the model's compute time does nothing about a 300 ms endpoint wait. Measure the stages, and report the hardware and chunk size with every latency number.

## 4 · Word and character error rates

Recognition is scored by the **edit distance** between the reference transcript and the hypothesis: the smallest number of substitutions $S$, deletions $D$ and insertions $I$ that turn one into the other. With $d_{i,j}$ the distance between the first $i$ reference tokens and the first $j$ hypothesis tokens,

$$d_{i,0} = i, \qquad d_{0,j} = j, \qquad d_{i,j} = \min\big(d_{i-1,j} + 1,\; d_{i,j-1} + 1,\; d_{i-1,j-1} + [r_i \ne h_j]\big),$$

where the three terms are a deletion, an insertion and a match or substitution. Tokens are words for the **word error rate** (WER) and characters for the **character error rate** (CER). Over a test set, add up the edits and the reference lengths before dividing:

$$\text{WER} = \frac{\sum_u (S_u + D_u + I_u)}{\sum_u N_u} .$$

Averaging per-utterance rates instead gives a one-word utterance as much weight as a fifty-word one. Because insertions are counted but the denominator is the reference length, WER can exceed 100%.

In [ ]:
reference_words, hypothesis_words = "the cat sat".split(), "the cat sat sat on".split()
print("reference:", reference_words, " hypothesis:", hypothesis_words)
print("one way to align them: 3 matches, then 2 insertions -> distance 2")

### ✍️ Your turn: `edit_distance`

Write `edit_distance(reference, hypothesis)` that returns the Levenshtein distance as an `int`, for any two sequences: lists of words or strings of characters.

Hint: keep only the previous row of the table, `previous = list(range(len(hypothesis) + 1))`, and build each new row starting from `[i]`.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def edit_distance_reference(reference, hypothesis):
    previous = list(range(len(hypothesis) + 1))
    for i, ref_token in enumerate(reference, start=1):
        current = [i]
        for j, hyp_token in enumerate(hypothesis, start=1):
            current.append(min(previous[j] + 1, current[j - 1] + 1, previous[j - 1] + (ref_token != hyp_token)))
        previous = current
    return previous[-1]

In [ ]:
def edit_distance(reference, hypothesis):
    """The fewest substitutions, deletions and insertions that turn reference into hypothesis."""
    # ✍️ Your code here (two nested loops over a table row)
    raise NotImplementedError

In [ ]:
edit_distance = practice("edit_distance", edit_distance, edit_distance_reference, cases=[
    (reference_words, hypothesis_words),
    ("kitten", "sitting"),                 # the classic example: 3
    ([], ["a", "b"]),                      # an empty reference: only insertions
    ("abc", "abc"),
    ("abcd", "bd"),                        # two deletions
])

### ✍️ Your turn: `error_rate`

Write `error_rate(references, hypotheses)` that returns the corpus-level rate: the sum of `edit_distance(reference, hypothesis)` over all pairs divided by the sum of the reference lengths. Each reference and hypothesis is a sequence (a list of words or a string of characters).

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def error_rate_reference(references, hypotheses):
    edits = sum(edit_distance(r, h) for r, h in zip(references, hypotheses))
    return edits / sum(len(r) for r in references)

In [ ]:
def error_rate(references, hypotheses):
    """Corpus-level error rate: total edits divided by total reference length."""
    # ✍️ Your code here (two lines)
    raise NotImplementedError

In [ ]:
error_rate = practice("error_rate", error_rate, error_rate_reference, cases=[
    (["abc", "ab"], ["abc", "ba"]),
    ([["ab"]], [["ab", "ab", "ab"]]),      # one reference word, three hypothesis words
    ([["the", "cat"], ["a"]], [["the", "hat"], []]),
])


def wer(references, hypotheses):
    return error_rate([r.split() for r in references], [h.split() for h in hypotheses])


def cer(references, hypotheses):
    """Character error rate over the letters, ignoring the spaces between words."""
    return error_rate([r.replace(" ", "") for r in references], [h.replace(" ", "") for h in hypotheses])


print("WER for 'ab' -> 'ab ab ab':", wer(["ab"], ["ab ab ab"]), "(two insertions over one reference word)")
refs, hyps = ["a", "abc ab ba cab ab abc"], ["", "abc ab ba cab ab abc"]
per_utterance = np.mean([wer([r], [h]) for r, h in zip(refs, hyps)])
print(f"one short utterance wrong, one long one right: corpus WER {wer(refs, hyps):.3f}, mean of per-utterance WERs {per_utterance:.3f}")

**Expected behaviour.** Two insertions over a one-word reference give a WER of 2.0, that is 200%. In the second example the corpus WER is $1/7 \approx 0.14$, while averaging per-utterance rates gives 0.5, because the one-word utterance counts as much as the six-word one.

## 5 · Put it together: an auditable speech experiment

Now join everything into one experiment, of the kind the capstone asks for. The plan is decided before any result is seen:

- **Data.** Utterances of one to three words from the lexicon, synthesised by `speak`. Each **voice** is a pitch factor and a speaking rate. Training uses three voices (pitch 0.9, 1.0 and 1.1) at 20 dB SNR. Testing uses voices that were never used in training, under four conditions: a new voice within the training range (pitch 0.95), the same voice with more noise (10 dB SNR), the same voice speaking faster (rate 0.7), and a new voice above the training range (pitch 1.25). The split unit is the voice.
- **Model.** A frame classifier on the tiny tensor engine: each log-mel frame with one frame of context on each side (60 inputs), 32 ReLU units and four outputs (pause, a, b, c), trained with cross-entropy on the frame labels that synthesis gives us for free.
- **Decoding.** Merge runs of the same letter; a pause of at least 6 frames (96 ms) separates words. This is the same collapse idea as CTC, with "pause" playing the role of the blank.
- **Streaming.** Test audio is processed by your `stream_frames` in 80 ms chunks.
- **Metrics.** CER and WER per condition with your `error_rate`; on the new-voice condition, the time from the onset of the first letter to the moment the system can emit it; and the measured real-time factor.

In [ ]:
def make_corpus(n, seed, voices, snr_db=20.0):
    """n utterances, each from a voice (pitch, rate) chosen in turn: texts, log-mel features, frame labels."""
    rng = np.random.default_rng(seed)
    texts, features, labels, voice_ids = [], [], [], []
    for u in range(n):
        voice_id = u % len(voices)
        pitch, rate = voices[voice_id]
        text = " ".join(rng.choice(LEXICON, size=int(rng.integers(1, 4))))
        x, frame_labels = speak(text, rng, pitch=pitch, rate=rate, snr_db=snr_db)
        texts.append(text); features.append(log_mel_spectrogram(x, SAMPLE_RATE)); labels.append(frame_labels); voice_ids.append(voice_id)
    return texts, features, labels, voice_ids


train_voices = [(0.9, 1.0), (1.0, 1.0), (1.1, 1.0)]
train_texts, train_feats, train_labels, train_voice_ids = make_corpus(150, seed=1, voices=train_voices)
feature_mean = np.concatenate(train_feats).mean(axis=0)              # training frames only
feature_std = np.concatenate(train_feats).std(axis=0) + 1e-8


def prepare(features, context=1):
    z = (features - feature_mean) / feature_std
    padded = np.pad(z, ((context, context), (0, 0)), mode="edge")
    return np.concatenate([padded[i:i + len(z)] for i in range(2 * context + 1)], axis=1)


train_inputs = np.concatenate([prepare(f) for f in train_feats])
train_targets = np.concatenate(train_labels)
print(f"{len(train_texts)} training utterances, {len(train_inputs)} labelled frames;",
      "frame classes (pause, a, b, c):", np.bincount(train_targets, minlength=4))

model_rng = np.random.default_rng(0)
W1, b1 = parameter(60, 32, rng=model_rng), zeros_parameter(32)
W2, b2 = parameter(32, 4, rng=model_rng, scale=0.1), zeros_parameter(4)
params = [W1, b1, W2, b2]


class Adam:
    """Adam on a list of engine parameters (as in Base Model Training)."""

    def __init__(self, params, learning_rate=0.01, beta1=0.9, beta2=0.999, eps=1e-8):
        self.params, self.learning_rate, self.beta1, self.beta2, self.eps = params, learning_rate, beta1, beta2, eps
        self.m = [np.zeros_like(p.data) for p in params]
        self.v = [np.zeros_like(p.data) for p in params]
        self.t = 0

    def step(self):
        self.t += 1
        for p, m, v in zip(self.params, self.m, self.v):
            m[:] = self.beta1 * m + (1 - self.beta1) * p.grad
            v[:] = self.beta2 * v + (1 - self.beta2) * p.grad ** 2
            m_hat = m / (1 - self.beta1 ** self.t)
            v_hat = v / (1 - self.beta2 ** self.t)
            p.data -= self.learning_rate * m_hat / (np.sqrt(v_hat) + self.eps)


optimizer, batches, frame_losses = Adam(params), np.random.default_rng(1), []
for step in range(200):
    rows = batches.choice(len(train_inputs), size=256, replace=False)
    loss = cross_entropy((Tensor(train_inputs[rows]) @ W1 + b1).relu() @ W2 + b2, train_targets[rows])
    zero_grad(params)
    loss.backward()
    optimizer.step()
    frame_losses.append(float(loss.data))
print(f"frame classifier: loss {frame_losses[0]:.3f} -> {frame_losses[-1]:.4f}")

In [ ]:
WORD_GAP_FRAMES, CHUNK_MS, LOOKAHEAD_FRAMES = 6, 80, 1


def classify_frames(features):
    return np.argmax(np.maximum(0, prepare(features) @ W1.data + b1.data) @ W2.data + b2.data, axis=1)


def frames_to_text(predictions):
    """Merge runs of a letter; a pause of at least WORD_GAP_FRAMES frames between letters starts a new word."""
    out, pause, previous = [], 0, 0
    for p in predictions:
        if p == 0:
            pause += 1
        else:
            if pause >= WORD_GAP_FRAMES and out:
                out.append(" ")
            if p != previous or pause > 0:
                out.append(LETTERS[p])
            pause = 0
        previous = p
    return "".join(out)


def recognise_streaming(x, chunk_ms=CHUNK_MS):
    """Stream x through your front end in chunks.

    Returns the text, the time in ms at which the first letter could be emitted (or None) and the compute time in s.
    """
    start = time.perf_counter()
    chunk = int(chunk_ms * SAMPLE_RATE / 1000)
    frames, ready = stream_frames([x[i:i + chunk] for i in range(0, len(x), chunk)], FRAME_LENGTH, HOP)
    power_frames = np.abs(np.fft.rfft(frames * hann_window(FRAME_LENGTH), axis=1)) ** 2
    predictions = classify_frames(log_mel(power_frames, mel_filterbank(N_MELS, FRAME_LENGTH, SAMPLE_RATE)))
    compute = time.perf_counter() - start
    letters = np.flatnonzero(predictions > 0)
    first_emitted_ms = None
    if len(letters):
        decision_frame = min(letters[0] + LOOKAHEAD_FRAMES, len(frames) - 1)   # lookahead: wait for the next frame
        first_emitted_ms = (ready[decision_frame] + 1) * chunk / SAMPLE_RATE * 1000
    return frames_to_text(predictions), first_emitted_ms, compute


test_conditions = {
    "new voice (pitch 0.95)": dict(voices=[(0.95, 1.0)], snr_db=20.0),
    "noisy (10 dB SNR)": dict(voices=[(0.95, 1.0)], snr_db=10.0),
    "fast (rate 0.7)": dict(voices=[(0.95, 0.7)], snr_db=20.0),
    "higher voice (pitch 1.25)": dict(voices=[(1.25, 1.0)], snr_db=20.0),
}
results, failures, audio_seconds, compute_seconds, latencies = {}, [], 0.0, 0.0, []
for c, (condition, settings) in enumerate(test_conditions.items()):
    rng = np.random.default_rng(100 + c)
    texts, hypotheses = [], []
    for u in range(40):
        pitch, rate = settings["voices"][0]
        text = " ".join(rng.choice(LEXICON, size=int(rng.integers(1, 4))))
        x, frame_labels = speak(text, rng, pitch=pitch, rate=rate, snr_db=settings["snr_db"])
        hypothesis, first_emitted_ms, compute = recognise_streaming(x)
        texts.append(text); hypotheses.append(hypothesis)
        audio_seconds += len(x) / SAMPLE_RATE; compute_seconds += compute
        if c == 0 and first_emitted_ms is not None:
            # the first letter's target begins at control frame m + 1, where m is its first analysis frame
            onset_ms = (np.flatnonzero(frame_labels > 0)[0] + 1) * HOP / SAMPLE_RATE * 1000
            latencies.append(first_emitted_ms - onset_ms)
        if hypothesis != text:
            failures.append((condition, text, hypothesis))
    results[condition] = {"CER": round(cer(texts, hypotheses), 3), "WER": round(wer(texts, hypotheses), 3), "utterances": len(texts)}
    print(f"{condition:26s}: CER {results[condition]['CER']:.3f}, WER {results[condition]['WER']:.3f}")
print(f"new voice: first letter emitted {np.mean(latencies):.0f} ms after its onset on average "
      f"(at most {np.max(latencies):.0f} ms), with {CHUNK_MS} ms chunks and {LOOKAHEAD_FRAMES} frame of lookahead")
print(f"real-time factor on this machine: {compute_seconds / audio_seconds:.4f} (below 1 is faster than real time)")
for condition, text, hypothesis in failures[:4]:
    print(f"  failure in '{condition}': said '{text}', recognised '{hypothesis}'")

In [ ]:
clean, noisy, fast, high = (results[name]["CER"] for name in test_conditions)
assert np.array_equal(stream_frames([stream_wave[i:i + 640] for i in range(0, len(stream_wave), 640)], FRAME_LENGTH, HOP)[0],
                      frame_signal(stream_wave, FRAME_LENGTH, HOP)), "streaming must reproduce the offline frames"
assert frame_losses[-1] < 0.25 * frame_losses[0], "the frame classifier should learn"
assert token_errors[32] < token_errors[2], "a larger codebook should reconstruct the frames better"
assert clean <= 0.05, f"a new voice inside the training range should be recognised, got CER {clean:.3f}"
assert noisy > clean and high > clean, "more noise and a voice outside the training range should both hurt"
assert all(latency > 0 for latency in latencies), "no letter can be emitted before it starts"
print(f"✅ Text -> speech -> streamed frames -> letters -> scores: CER {clean:.3f} on a new voice in range, "
      f"{noisy:.3f} at 10 dB SNR, {fast:.3f} when fast and {high:.3f} for the higher voice.")

**What to notice.** The recogniser handles a new voice inside the training range, but the failure slices show what one aggregate number would hide. At 10 dB SNR the recogniser hears noise in the pauses as the high letter c and inserts it, and a CER above 0.2 follows from a change the clean score could not predict. A fast speaker's pauses become shorter than the word-gap rule expects, so words run together and WER suffers more than CER. A voice 25% higher moves every letter's pitch into a band that belonged to another letter in training, so letters are confused systematically. Each failure suggests its own controlled follow-up: noise augmentation, a rate-aware word-gap rule, or pitch augmentation by speed perturbation, as in notebook 03.

## 6 · The capstone scaffold

The capstone asks for an experiment that someone else could audit. The cell below gathers this notebook's run into exactly that record: a manifest, the signal and model configuration, the decoding and streaming settings, metrics by condition, latency, failures and limitations. Then the **Your capstone** list says how to extend it.

In [ ]:
run_report = {
    "task": "recognise generated letter tones (a, b, c) in words; not human speech",
    "manifest": {
        "split_unit": "voice (pitch factor, speaking rate)",
        "train": {"voices": [f"pitch {p}, rate {r}" for p, r in train_voices], "utterances": len(train_texts),
                  "snr_db": 20.0, "seed": 1},
        "test": {name: {"voices": [f"pitch {p}, rate {r}" for p, r in settings["voices"]], "snr_db": settings["snr_db"],
                        "utterances": results[name]["utterances"], "seed": 100 + c}
                 for c, (name, settings) in enumerate(test_conditions.items())},
    },
    "signal": {"sample_rate": SAMPLE_RATE, "channels": "mono", "frame_length": FRAME_LENGTH, "hop": HOP,
               "window": "periodic Hann", "features": f"log-mel, {N_MELS} HTK bands, eps 1e-10",
               "normalisation": "per-band mean and std of the training frames"},
    "text_normalisation": "lower-case letters a, b and c; words separated by single spaces",
    "model": {"type": "frame classifier 60-32-4 (ReLU), tiny tensor engine", "parameters": int(sum(p.data.size for p in params)),
              "optimiser": "Adam, learning rate 0.01, 200 steps of 256 frames", "context_frames": 1},
    "decoding": {"rule": "merge runs, pauses separate letters", "word_gap_frames": WORD_GAP_FRAMES},
    "streaming": {"chunk_ms": CHUNK_MS, "lookahead_frames": LOOKAHEAD_FRAMES,
                  "mean_first_letter_latency_ms": round(float(np.mean(latencies)), 1),
                  "real_time_factor": round(compute_seconds / audio_seconds, 4), "hardware": "this notebook's machine"},
    "metrics": results,
    "failures": [{"condition": c, "reference": t, "hypothesis": h} for c, t, h in failures[:5]],
    "limitations": "Synthetic tones from one oscillator design; 40 test utterances per condition; one seed; "
                   "no listening test, no real speakers and no claim about speech recognition.",
}
required = ["task", "manifest", "signal", "text_normalisation", "model", "decoding", "streaming", "metrics", "failures", "limitations"]
assert all(run_report.get(key) for key in required), "every section of the report needs content"
print(json.dumps(run_report, indent=1))

### Your capstone

Use this scaffold for the course capstone. Work in a copy of this notebook and change **one thing at a time**, so that every difference in the report has one cause.

1. **State a question before you run anything**, for example "Does speed-perturbation augmentation of the training voices fix the higher-voice condition without hurting the others?", "How does first-letter latency change with 20, 80 and 200 ms chunks?" or "Does noise augmentation at 5 to 20 dB SNR help at 10 dB?".
2. **Keep the test conditions fixed** and their seeds unchanged. If you need to choose a setting (an augmentation range, a word-gap threshold), choose it on a validation voice that is not one of the test voices, and record it in the manifest.
3. **Re-run and update `run_report`**: the configuration you changed, the metrics for every condition (not only the one you hoped to improve), latency, and three failures that you have looked at, by listening as well as reading.
4. **Write the conclusion with its limits**, in the form "Under these conditions, with this many utterances and this seed, X changed by Y; we have not tested Z."

If you go further, with consented recordings and a pretrained model in an environment that has a GPU and the right licences, keep the same structure: a manifest with speakers and recording groups and their split IDs, the sample-rate and feature configuration, the decoding or synthesis settings, metrics by condition, listening notes, latency and a failure analysis.

## Exercises

### Exercise 1: `edit_operations`

Extend the edit distance to count each kind of error. Write `edit_operations(reference, hypothesis)` that returns `(substitutions, deletions, insertions)` for one minimal alignment. Keep the whole table, then trace back from the bottom-right corner, preferring a match or substitution, then a deletion, then an insertion whenever several moves are optimal. (Only the total is unique in general; the check uses cases whose split is unambiguous.)

In [ ]:
def edit_operations_reference(reference, hypothesis):
    n, m = len(reference), len(hypothesis)
    d = np.zeros((n + 1, m + 1), dtype=int)
    d[:, 0], d[0, :] = np.arange(n + 1), np.arange(m + 1)
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            d[i, j] = min(d[i - 1, j] + 1, d[i, j - 1] + 1, d[i - 1, j - 1] + (reference[i - 1] != hypothesis[j - 1]))
    i, j, subs, dels, ins = n, m, 0, 0, 0
    while i > 0 or j > 0:
        if i > 0 and j > 0 and d[i, j] == d[i - 1, j - 1] + (reference[i - 1] != hypothesis[j - 1]):
            subs += int(reference[i - 1] != hypothesis[j - 1]); i, j = i - 1, j - 1
        elif i > 0 and d[i, j] == d[i - 1, j] + 1:
            dels += 1; i -= 1
        else:
            ins += 1; j -= 1
    return subs, dels, ins

In [ ]:
def edit_operations(reference, hypothesis):
    """(substitutions, deletions, insertions) along one minimal alignment."""
    # ✍️ Your code here
    raise NotImplementedError

In [ ]:
edit_operations = practice("edit_operations", edit_operations, edit_operations_reference, cases=[
    ("abc", "abd"), ("abcd", "abd"), ("ab", "abxy"), ("the cat sat".split(), "the hat".split()),
])

### Exercise 2: `bits_per_second`

Write `bits_per_second(frame_rate, codebook_size, n_codebooks=1)` for a token stream in which every frame carries one index from each of `n_codebooks` codebooks. How many bits per second does a codec with 75 frames per second and 8 codebooks of 1,024 entries need, and how does that compare with 16-bit PCM at 24 kHz?

In [ ]:
def bits_per_second_reference(frame_rate, codebook_size, n_codebooks=1):
    return float(frame_rate * n_codebooks * np.log2(codebook_size))

In [ ]:
def bits_per_second(frame_rate, codebook_size, n_codebooks=1):
    """Bit rate of a stream of codebook indices."""
    # ✍️ Your code here
    raise NotImplementedError

In [ ]:
bits_per_second = practice("bits_per_second", bits_per_second, bits_per_second_reference, cases=[
    (62.5, 16), (75.0, 1024, 8), (50.0, 2),
])

### Exercise 3: `endpoint_frame`

An endpointer declares the end of an utterance after enough consecutive non-speech frames *following* some speech. Write `endpoint_frame(is_speech, silence_frames)` that returns the index of the frame at which the endpoint is declared (the last frame of the first run of `silence_frames` non-speech frames that comes after at least one speech frame), or `-1` if that never happens. With a 16 ms hop, how much latency does `silence_frames=19` add to every final transcript?

In [ ]:
def endpoint_frame_reference(is_speech, silence_frames):
    heard_speech, run = False, 0
    for index, speech in enumerate(is_speech):
        if speech:
            heard_speech, run = True, 0
        elif heard_speech:
            run += 1
            if run == silence_frames:
                return index
    return -1

In [ ]:
def endpoint_frame(is_speech, silence_frames):
    """Index of the frame where enough silence after speech has been seen, or -1."""
    # ✍️ Your code here
    raise NotImplementedError

In [ ]:
endpoint_frame = practice("endpoint_frame", endpoint_frame, endpoint_frame_reference, cases=[
    ([0, 0, 1, 1, 0, 0, 0, 1], 3),        # 6: three frames of silence after speech
    ([0, 0, 0, 0], 2),                    # no speech yet, so no endpoint: -1
    ([1, 0, 1, 0, 0], 2),                 # the pause at frame 1 is too short
])

### Exercise 4: an open question

Replace the recogniser's continuous features by tokens: map each of the 16 codes above to the frame label it most often carries in the training corpus, decode test utterances through the codes, and measure CER as the codebook grows from 2 to 32. Where does recognition stop improving, and how does that compare with where the reconstruction error stops improving? What does this suggest about choosing a codebook for a model that must both recognise and resynthesise speech?

## Common errors and debugging

- **Clicks or buzz in synthesised audio.** The phase was computed as $2\pi f_0[n]\,n / f_s$ instead of being accumulated, or the amplitude jumps from frame to frame instead of being interpolated.
- **k-means codes that never change or contain `nan`.** An empty cluster's mean is `nan`; keep its old vector, and initialise from distinct frames.
- **Streamed frames that differ slightly from offline frames.** The buffer dropped a whole frame instead of one hop after emitting it, or a chunk was skipped when it was shorter than a frame.
- **A WER above 1.** Not necessarily a bug: count the insertions. A WER that is too *low* is more suspicious: check that you divide by the reference length, not the hypothesis length.
- **Latency claims without conditions.** State the chunk size, lookahead, endpointing rule and hardware; a model that needs future audio cannot be called streaming just because it is fast.
- **Test conditions tuned on.** If a setting was chosen by looking at the test conditions, they are no longer an independent test.

## Summary

Speech synthesis runs the recognition pipeline backwards: a frontend and an acoustic model produce frame-level targets, and a vocoder, here an oscillator bank with an accumulated phase, turns them into samples. Audio tokens come from quantising frames with a codebook; the codebook size sets the bit rate and the reconstruction error. A streaming front end reproduces offline frames exactly while every chunk, window, lookahead and endpoint adds latency. Recognition is scored with corpus-level WER and CER from an edit distance, and the results only mean something with their conditions attached, which is what the capstone report records.

## Next

You have built the whole speech stack in miniature: features, CTC and attention-based recognition, data splits and augmentation, synthesis, tokens, streaming and evaluation. Complete the capstone above. From here, the **ML System Design** course looks at how such components are deployed, monitored and budgeted as a service.